## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import scipy.sparse as sps

from Challenge.paths import load_holdout_split, save_xgboost_cv_folds

np.random.seed(42)

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
URM_inner, _ = load_holdout_split()

## **Create K Cross validation**

In [5]:
# Shuffle the indices for splitting
indices = np.arange(0, URM_inner.nnz, 1)
np.random.shuffle(indices)

In [6]:
folds_5_idx = np.array_split(indices, 5)

In [7]:
def get_sparse_from_indices(URM, indices):
    """
    Given a URM and a list of indices, return a sparse matrix containing only the interactions
    corresponding to the given indices.
    """
    URM_coo = URM.tocoo()
    
    rows = URM_coo.row[indices]
    cols = URM_coo.col[indices]
    data = URM_coo.data[indices]
    
    return sps.coo_matrix((data, (rows, cols)), shape=URM.shape).tocsr()


In [8]:
# Create the train-validation splits for each fold
folds_5 = []
for fold_indices in folds_5_idx:
    val_idx = fold_indices
    train_idx = np.setdiff1d(indices, val_idx)
    
    URM_validation = get_sparse_from_indices(URM_inner, val_idx)
    URM_train = get_sparse_from_indices(URM_inner, train_idx)
    
    folds_5.append((URM_train, URM_validation))

In [9]:
# Check splits
for i, (URM_train, URM_validation) in enumerate(folds_5):
    print(f"Fold {i+1}")
    print("URM_all:", URM_inner.shape)
    print("URM_train:", URM_train.shape)
    print("URM_validation:", URM_validation.shape)
    print()
    print("URM_all:", len(URM_inner.nonzero()[0]))
    print("URM_train:", len(URM_train.nonzero()[0]))
    print("URM_validation:", len(URM_validation.nonzero()[0]))
    print("-" * 30)

Fold 1
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 1947556
URM_validation: 486890
------------------------------
Fold 2
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 1947557
URM_validation: 486889
------------------------------
Fold 3
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 1947557
URM_validation: 486889
------------------------------
Fold 4
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 1947557
URM_validation: 486889
------------------------------
Fold 5
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 1947557
URM_validation: 486889
------------------------------


In [10]:
# Save the folds
save_xgboost_cv_folds(folds_5)